In [1]:
from Data_Preparation import Dataset
from Data_Preparation.Embedding import embedding_encoder 
from Data_Preparation.Tac import tac
from Modelisation.Baselines.OCSVM import ocsvm
from torch.utils.data import ConcatDataset, DataLoader
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net, embedding_layer
import Modelisation.Baselines.CVDD.networks.utils as utils
import torch.optim as optim
import numpy as np
import time
import torch
from sklearn.cluster import KMeans
from datasets import concatenate_datasets
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.model_selection import KFold

## Import

In [ ]:
import os
import re

def save_results(dataset_name, inlier_topic, type_emb, auc, ap, fpr95,
                 output_dir="/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Results",
                 filename="results.txt", overwrite=False):

    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)

    existing_content = ""
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            existing_content = f.read()

    pattern = (
        rf"Dataset:\s*{re.escape(dataset_name)}\s*"
        rf"Inlier class:\s*{re.escape(inlier_topic)}\s*"
        rf"Embedding type:\s*{re.escape(type_emb)}"
    )

    new_block = (
        "========================================\n"
        f"Dataset:        {dataset_name}\n"
        f"Inlier class:   {inlier_topic}\n"
        f"Embedding type: {type_emb}\n"
        "----------------------------------------\n"
        f"AUC:            {auc:.4f}\n"
        f"Avg Precision:  {ap:.4f}\n"
        f"FPR@95:         {fpr95:.4f}\n"
        "========================================\n\n"
    )

    if re.search(pattern, existing_content):
        if overwrite:
            existing_content = re.sub(
                r"========================================\n"
                + pattern
                + r".*?========================================\n\n",
                new_block,
                existing_content,
                flags=re.DOTALL
            )
            print(f"Résultats mis à jour pour ({dataset_name}, {inlier_topic}, {type_emb}).")
        else:
            print(f"Résultats déjà présents, non modifiés : ({dataset_name}, {inlier_topic}, {type_emb}).")
            return
    else:
        existing_content += new_block
        print(f"Nouveaux résultats ajoutés pour ({dataset_name}, {inlier_topic}, {type_emb}).")

    with open(filepath, "w") as f:
        f.write(existing_content)


## Test

In [ ]:
def ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid, n_splits=3, verbose=True):
    best_auc = -np.inf
    best_params = None

    for kernel in param_grid["kernel"]:
        for nu in param_grid["nu"]:
            for gamma in param_grid["gamma"]:
                clf = OneClassSVM(kernel=kernel, nu=nu, gamma=gamma)
                clf.fit(X_train)

                scores_test = -clf.decision_function(X_test)
                auc = roc_auc_score(y_true_test, scores_test)

                # print(f"{kernel}, nu={nu}, gamma={gamma} -> AUC={auc:.4f}")

                if auc > best_auc:
                    best_auc = auc
                    best_params = {"kernel": kernel, "nu": nu, "gamma": gamma}

    print("\n✅ Best params:", best_params, " | AUC =", best_auc)
    return best_params, best_auc

In [ ]:
dataset_name = '20newsgroups'
full_dataset_ = False
preprocessing = True

dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
trainset, testset = dataset.get_splits()

type_tac = "ruff"
anomaly_rate = 0.1


emb_model = 'glove_300d.kv'
type_emb = 'glove'

emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)

inlier_topic = "science"
# list_inlier_topic = ["earn", "acq", "crude", "trade", "money-fx", "interest", "ship"]


# for inlier_topic in list_inlier_topic:
# print(inlier_topic)
inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

data_test = tac.textual_anomaly_contamination(
                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)

inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)
data_test_emb = emb_encoder.forward(data_test)


X_train = np.array(inlier_dataset_train_emb[f'{type_emb}_embedding'])
X_test = np.array(data_test_emb[f'{type_emb}_embedding'])
y_true_test = np.array(data_test_emb['anomaly_class'])  
y_true_train = np.array(anomaly_dataset_train['anomaly_class'])  


In [ ]:
param_grid = {
    "kernel": ["rbf", "sigmoid"],
    "nu": [0.05, 0.1, 0.15],
    "gamma": [1e-3, 1e-2, 1e-1, 1]
}

best_params, results = ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid, n_splits=3)
print(best_params, end="\n")
# clf = OneClassSVM(**best_params)
clf, _, _ = ocsvm.One_Class_SVM(inlier_dataset_train_emb[f'{type_emb}_embedding'], best_params)
print(clf)
clf.fit(X_train)

scores_test = clf.decision_function(X_test)
# print(scores_test)
auc, ap, fpr95 = ev.evaluation(y_true_test, scores_test, verbose=True)
print("Final results with best params:", auc, ap, fpr95)

In [ ]:
# ==============================
# ------------ DATA ------------
# ==============================

list_dataset_name = ['20newsgroups', 'reuters']
list_list_inlier_topic = [
    ["computer", "recreation", "science", "miscellaneous", "politics", "religion"],
    ["earn", "acq", "crude", "trade", "money-fx", "interest", "ship"]
]
for dataset_name, list_inlier_topic in zip(list_dataset_name, list_list_inlier_topic):

    full_dataset_ = False
    preprocessing = True

    dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
    trainset, testset = dataset.get_splits()

    type_tac = "ruff"
    anomaly_rate = 0.1

    # ==============================
    # --------- EMBEDDING ----------
    # ==============================

    list_emb_model = ['glove_300d.kv', 'fasttext_300d.kv']
    list_type_emb = ['glove', 'fasttext']
    # emb_model = "distilbert-base-uncased"
    # type_emb = "bert"

    for emb_model, type_emb in zip(list_emb_model, list_type_emb):

        emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)


        # ==============================
        # ------------ LOOP ------------
        # ==============================

        for inlier_topic in list_inlier_topic:
            print(inlier_topic)
            inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

            data_test = tac.textual_anomaly_contamination(
                                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)
            print(inlier_dataset_train.shape)
            print(anomaly_dataset_train.shape)
            print(data_test.shape)
            print("------------------------------------")

            inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)
            data_test_emb = emb_encoder.forward(data_test)

            print(inlier_dataset_train_emb[f'{type_emb}_embedding'])

            # ocsvm_kwargs = {
            #     "nu": 0.4,
            #     "kernel": 'rbf',
            #     "gamma": 'scale'
            # }
            # clf, y_pred_train, scores_train = ocsvm.One_Class_SVM(inlier_dataset_train_emb[f'{type_emb}_embedding'], ocsvm_kwargs)

            # y_pred_test = clf.predict(data_test_emb[f'{type_emb}_embedding'])           
            # scores_test = clf.decision_function(data_test_emb[f'{type_emb}_embedding'])

            # auc, ap, fpr95 = ev.evaluation(data_test_emb['anomaly_class'], scores_test, verbose=False)
            
            # output_dir = "/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Results"
            # save_results(dataset_name, inlier_topic, type_emb, auc, ap, fpr95, output_dir=output_dir, filename="results.txt", overwrite=False)
            print("===============================================\n")



## OCSVM

In [ ]:
emb_model = "glove_300d.kv"
type_emb = "glove"

# emb_model = "fasttext_300d.kv"
# type_emb = "fasttext"

# emb_model = "distilbert-base-uncased"
# type_emb = "bert"

emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)

In [ ]:
inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)

In [ ]:
inlier_dataset_train_emb['glove_embedding']

In [ ]:
ocsvm_kwargs = {
        "nu": 0.4,
        "kernel": 'rbf',
        "gamma": 'scale'
    }
clf, y_pred_train, scores_train = ocsvm.One_Class_SVM(inlier_dataset_train_emb['glove_embedding'],
                                                      ocsvm_kwargs
                                                         )

In [ ]:
ds = ConcatDataset([emb_encoder.forward(inlier_dataset_test), emb_encoder.forward(anomaly_dataset_test)])
ds

In [ ]:
inputs_test = [x['glove_embedding'] for x in ds]
labels_test = [y['anomaly_class'] for y in ds]

y_pred_test = clf.predict(inputs_test)           
scores_test = clf.decision_function(inputs_test)

auc, f1, precision, recall, fpr95 = ev.evaluation(labels_test, scores_test, y_pred_test, verbose=False)

print(clf, end="\n\n")

print(auc)
print(f1)
print(precision)
print(recall)
print(fpr95)


## CVDD

### loop

In [2]:
def initialize_context_vectors(net, train_loader):
    """
    Initialize the context vectors from an initial run of k-means++ on simple average sentence embeddings

    Returns
    -------
    centers : ndarray, [n_clusters, n_features]
    """

    # Get vector representations
    X = ()
    for data in train_loader:
        inputs, _, _, _ = data
        # text.shape = (sentence_length, batch_size)

        X_batch = net.pretrained_model(inputs)
        # X_batch.shape = (sentence_length, batch_size, embedding_size)

        # compute mean and normalize
        X_batch = torch.mean(X_batch, dim=0)
        X_batch = X_batch / torch.norm(X_batch, p=2, dim=1, keepdim=True).clamp(min=1e-08)
        X_batch[torch.isnan(X_batch)] = 0
        # X_batch.shape = (batch_size, embedding_size)

        X += (X_batch.cpu().data.numpy(),)

    X = np.concatenate(X)
    n_attention_heads = net.n_attention_heads

    kmeans = KMeans(n_clusters=n_attention_heads).fit(X)
    centers = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, ord=2, axis=1, keepdims=True)


    return centers

In [5]:
dataset_name = 'reuters'
full_dataset_ = False
preprocessing = True

dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
trainset, testset = dataset.get_splits()

inlier_topic = "acq"
type_tac = "ruff"
anomaly_rate = 0.1

inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

data_test = tac.textual_anomaly_contamination(
                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)

In [6]:
corpus = inlier_dataset_train['text']
vocab = utils.build_vocab(corpus,min_freq=1)
seq_len = 200

cvdd_dataset_train = Dataset.ADdatasets.CVDDDatasetWrapper(inlier_dataset_train, embedding_type='glove', vocab=vocab, seq_len=seq_len)
cvdd_dataset_test = Dataset.ADdatasets.CVDDDatasetWrapper(data_test, embedding_type='glove', vocab=vocab, seq_len=seq_len)

pretrained_model = embedding_layer.EmbeddingFactory.create('glove',
                        glove_path='./Modelisation/Baselines/CVDD/embedding_models/glove.6B.300d.txt',
                        vocab=vocab,
                        embedding_dim=300,
                        trainable=True)

In [ ]:
dl_train = DataLoader(cvdd_dataset_train, batch_size=64, shuffle=True)
dl_test = DataLoader(cvdd_dataset_test, batch_size=64, shuffle=True)

attention_size = 150 
n_attention_heads = 4
model = cvdd_Net.CVDDNet(pretrained_model, attention_size, n_attention_heads)

In [ ]:
alpha_scheduler = 'logarithmic'
n_epochs = 100

# alpha annealing strategy
alpha_milestones = np.arange(1, 6) * int(n_epochs / 5)  # 5 equidistant milestones over n_epochs
if alpha_scheduler == 'soft':
    alphas = [0.0] * 5
if alpha_scheduler == 'linear':
    alphas = np.linspace(.2, 1, 5)
if alpha_scheduler == 'logarithmic':
    alphas = np.logspace(-4, 0, 5)
if alpha_scheduler == 'hard':
    alphas = [100.0] * 4

In [9]:
lr = 0.01
weight_decay = 1e-6
lr_milestones = [40,60]
lambda_p = 1.0

train_dists = None
train_att_matrix = None
train_top_words = None
c = None


In [10]:
model.c.data = torch.from_numpy(
                initialize_context_vectors(model, dl_train)[np.newaxis, :])

parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(parameters, lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=lr_milestones, gamma=0.1)

In [11]:
model.train()
alpha_i = 0

for epoch in range(n_epochs):
    
    scheduler.step()

    if epoch in lr_milestones:
        print(f"LR scheduler: new learning rate is %g" % float(scheduler.get_last_lr()[0]))

    if epoch in alpha_milestones:
        model.alpha = float(alphas[alpha_i])
        print('  Temperature alpha scheduler: new alpha is %g' % model.alpha)
        alpha_i += 1

    epoch_loss = 0.0
    n_batches = 0
    att_matrix = np.zeros((n_attention_heads, n_attention_heads))
    dists_per_head = ()
    epoch_start_time = time.time()

    for inputs, labels, texts, idx in dl_train:

        inputs = inputs.transpose(0, 1)
        
        optimizer.zero_grad() 

        cosine_dists, context_weights, A = model(inputs)
        scores = context_weights * cosine_dists

        I = torch.eye(n_attention_heads)
        CCT = model.c @ model.c.transpose(1, 2)
        P = torch.mean((CCT.squeeze() - I) ** 2)


        loss_P = lambda_p * P
        loss_emp = torch.mean(torch.sum(scores, dim=1))
        loss = loss_emp + loss_P


        dists_per_head += (cosine_dists.cpu().data.numpy(),)


        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)  # clip gradient norms in [-0.5, 0.5]
        optimizer.step()

        AAT = A @ A.transpose(1, 2)
        att_matrix += torch.mean(AAT, 0).cpu().data.numpy()

        epoch_loss += loss.item()
        n_batches += 1


    epoch_train_time = time.time() - epoch_start_time
    print(f'| Epoch: {epoch + 1:03}/{n_epochs:03} | Train Time: {epoch_train_time:.3f}s '
    f'| Train Loss: {epoch_loss / n_batches:.6f} |')


    train_dists = np.concatenate(dists_per_head)
    train_att_matrix = att_matrix / n_batches
    train_att_matrix = train_att_matrix.tolist()


c = np.squeeze(model.c.data.numpy())
c = c.tolist()


/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/lib/python3.12/site-packages/torch/autograd/graph.py:841: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  re

| Epoch: 001/060 | Train Time: 5.581s | Train Loss: 0.133277 |
| Epoch: 002/060 | Train Time: 6.538s | Train Loss: 0.029971 |
| Epoch: 003/060 | Train Time: 5.122s | Train Loss: 0.026365 |
| Epoch: 004/060 | Train Time: 5.685s | Train Loss: 0.025737 |
| Epoch: 005/060 | Train Time: 5.703s | Train Loss: 0.025530 |
| Epoch: 006/060 | Train Time: 5.767s | Train Loss: 0.025377 |
| Epoch: 007/060 | Train Time: 6.287s | Train Loss: 0.025217 |
| Epoch: 008/060 | Train Time: 5.517s | Train Loss: 0.025210 |
| Epoch: 009/060 | Train Time: 5.825s | Train Loss: 0.025269 |
| Epoch: 010/060 | Train Time: 5.746s | Train Loss: 0.025308 |
| Epoch: 011/060 | Train Time: 5.249s | Train Loss: 0.025262 |
| Epoch: 012/060 | Train Time: 5.401s | Train Loss: 0.025227 |
  Temperature alpha scheduler: new alpha is 0.0001
| Epoch: 013/060 | Train Time: 5.407s | Train Loss: 0.025638 |
| Epoch: 014/060 | Train Time: 5.519s | Train Loss: 0.025226 |
| Epoch: 015/060 | Train Time: 5.109s | Train Loss: 0.025391 |
| Ep

In [26]:
test_dists = None
test_att_matrix = None
test_top_words = None
test_auc = 0.0
test_scores = None
test_att_weights = None

ad_score = 'context_dist_mean'

In [55]:
n_attention_heads = model.n_attention_heads
epoch_loss = 0.0
n_batches = 0
att_matrix = np.zeros((n_attention_heads, n_attention_heads))
dists_per_head = ()
idx_label_score_head = []
att_weights = []
start_time = time.time()
model.eval()

with torch.no_grad():
    for inputs, labels, texts, idx in dl_test:

        cosine_dists, context_weights, A = model(inputs)
        scores = context_weights * cosine_dists
        _, best_att_head = torch.min(scores, dim=1)

        I = torch.eye(n_attention_heads)
        CCT = model.c @ model.c.transpose(1, 2)
        P = torch.mean((CCT.squeeze() - I) ** 2)

        loss_P = lambda_p * P
        loss_emp = torch.mean(torch.sum(scores, dim=1))
        loss = loss_emp + loss_P

        dists_per_head += (cosine_dists.cpu().data.numpy(),)
        ad_scores = torch.mean(cosine_dists, dim=1)

        idx_label_score_head += list(zip(idx,
                                    labels.cpu().data.numpy().tolist(),
                                    ad_scores.cpu().data.numpy().tolist(),
                                    best_att_head.cpu().data.numpy().tolist()))
        
        att_weights += A[best_att_head][:][range(len(idx))].cpu().data.numpy().tolist()

        AAT = A @ A.transpose(1, 2)
        att_matrix += torch.mean(AAT, 0).cpu().data.numpy()

        epoch_loss += loss.item()
        n_batches += 1


test_dists = np.concatenate(dists_per_head)
test_att_matrix = att_matrix / n_batches
test_att_matrix = test_att_matrix.tolist()

test_scores = idx_label_score_head
test_att_weights = att_weights

# Compute AUC
_, labels, scores, _ = zip(*idx_label_score_head)
labels = np.array(labels)
scores = np.array(scores)

if np.sum(labels) > 0:
    best_context = None
    if ad_score == 'context_dist_mean':
        test_auc = roc_auc_score(labels, scores)
    if ad_score == 'context_best':
        test_auc = 0.0
        for context in range(n_attention_heads):
            auc_candidate = roc_auc_score(labels, test_dists[:, context])
            print(auc_candidate)
            if auc_candidate > test_auc:
                test_auc = auc_candidate
                best_context = context
            else:
                pass
else:
    best_context = None
    test_auc = 0.0


# Log results
print('Test Loss: {:.6f}'.format(epoch_loss / n_batches))
print('Test AUC: {:.2f}%'.format(100. * test_auc))
print(f'Test Best Context: {best_context}')
print('Finished testing.')


Test Loss: 0.132740
Test AUC: 38.57%
Test Best Context: None
Finished testing.


### inference

In [ ]:
cvdd_model_pipeline(data_train, data_test, attention_size, n_attention_heads, embedding_type, seq_len, batch_size, shuffle, tokenizer=None, vocab=None)

In [ ]:
type_emb = 'glove'

In [ ]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(args.emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = data_train['text']
    vocab = utils.build_vocab(corpus,min_freq=1)
    tokenizer = None

model, dl_train, dl_test = cvdd_model_pipeline(data_train, data_test, args.attention_size, args.n_attention_heads, args.type_emb, 200, args.batch_size, args.shuffle, tokenizer, vocab)

cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=1e-2, lr_milestones=(20,25), n_epochs=30, 
            lambda_p=0.0, alpha_scheduler='linear', weight_decay=1e-6)

model_trained = cvdd_trainer.train(model, dl_train)
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')

save_results(args, auc, ap, fpr95,
            output_dir="/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Anomaly Detection Framework/Results",
            filename="results.txt",
            overwrite="smart")

## Save

In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM as SklearnOCSVM
from pyod.models.ocsvm import OCSVM as PyODOCSVM
from sklearn.metrics import roc_auc_score

# --- Création d'un dataset toy ---
np.random.seed(42)
X_train = 0.3 * np.random.randn(100, 2)           # données normales
X_test = np.r_[X_train, 2 + 0.3 * np.random.randn(20, 2)]  # ajouter anomalies
y_test = np.array([0]*100 + [1]*20)               # 0 = normal, 1 = anomalie

# --- Hyperparamètres communs ---
kernel = 'rbf'
gamma = 0.1
nu = 0.1  # proportion d'anomalies attendues

# --- Sklearn OCSVM ---
sk_ocsvm = SklearnOCSVM(kernel=kernel, gamma=gamma, nu=nu)
sk_ocsvm.fit(X_train)
sk_scores = -sk_ocsvm.decision_function(X_test)  # négatif car dans sklearn, + = normal, - = anomalie
sk_auc = roc_auc_score(y_test, sk_scores)

# --- PyOD OCSVM ---
pyod_ocsvm = PyODOCSVM(kernel=kernel, gamma=gamma, nu=nu)
pyod_ocsvm.fit(X_train)
pyod_scores = pyod_ocsvm.decision_function(X_test)  # déjà + = anomalie
pyod_auc = roc_auc_score(y_test, pyod_scores)

# --- Résultats ---
print(f"ROC AUC Sklearn OCSVM: {sk_auc:.4f}")
print(f"ROC AUC PyOD OCSVM:    {pyod_auc:.4f}")
